# Stage 9B — Resumable daily backfill

**Goal:** Collect every day in the Stage 9A plan from NYC Open Data. Each day gets its own data file and checksum manifest. Reruns skip days with manifests; the final dataset step re-verifies every saved file.

**Why:** Our first model missed winter patterns partly because one day per month did not cover them. A daily collection removes that sampling gap. It does not by itself guarantee a good model.

**Time:** Cell 7 runs for up to about 10 minutes and prints each finished day. It may take several runs to reach all 485 days. Rerun Cells 2–8 after a Colab restart; never change the plan ID. Network or source changes can cause a day to fail visibly, and rerunning retries that day. We will not inspect test outcomes while collecting them.

In [ ]:
# Cell 2 — Why: load the exact approved coverage plan.
from google.colab import drive
from pathlib import Path
from datetime import date, datetime, timedelta, timezone
import hashlib, json, os, requests, time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
plan_id = "20260926T130607938949Z"
plan_file = root / "release_decisions" / plan_id / "coverage_plan.json"
plan = json.loads(plan_file.read_text())
if plan["plan_id"] != plan_id or plan["daily_partitions_planned"] != 485:
    raise ValueError("Wrong coverage plan")
daily_root = root / "daily_backfill" / plan_id
daily_root.mkdir(parents=True, exist_ok=True)
print("Plan:", plan_id, "Days:", plan["daily_partitions_planned"])

In [ ]:
# Cell 3 — Why: retry temporary API failures and count a day first.
session = requests.Session()
retry = Retry(total=4, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=["GET"])
session.mount("https://", HTTPAdapter(max_retries=retry))
token = os.environ.get("SOCRATA_APP_TOKEN")
if token:
    session.headers["X-App-Token"] = token
page_size = 5000

def query_day(day):
    following = (date.fromisoformat(day) + timedelta(days=1)).isoformat()
    return (f"created_date >= '{day}T00:00:00' AND "
            f"created_date < '{following}T00:00:00'")

def count_day(where):
    response = session.get(plan["source_url"],
                           params={"$select": "count(*) as n", "$where": where}, timeout=60)
    response.raise_for_status()
    return int(response.json()[0]["n"])

print("Collector ready; page size:", page_size)

In [ ]:
# Cell 4 — Why: page in a fixed order and detect a changing source.
def fetch_day(day):
    where = query_day(day)
    expected = count_day(where)
    rows = []
    for offset in range(0, expected, page_size):
        params = {"$select": ",".join(plan["fields"]), "$where": where,
                  "$order": "created_date ASC, unique_key ASC",
                  "$limit": min(page_size, expected - offset), "$offset": offset}
        response = session.get(plan["source_url"], params=params, timeout=120)
        response.raise_for_status()
        page = response.json()
        if len(page) != params["$limit"]:
            raise ValueError(f"{day}: incomplete page at offset {offset}")
        rows.extend(page)
    if count_day(where) != expected or len(rows) != expected:
        raise ValueError(f"{day}: source row count changed during download")
    keys = [row.get("unique_key") for row in rows]
    if None in keys or len(keys) != len(set(keys)):
        raise ValueError(f"{day}: missing or duplicate request ID")
    if any(row.get("created_date", "")[:10] != day for row in rows):
        raise ValueError(f"{day}: request outside selected date")
    return rows, where, expected

print("Full-day paging function ready")

In [ ]:
# Cell 5 — Why: save a day only after its data and checksum pass.
def save_day(day):
    folder = daily_root / day
    data_file, meta_file = folder / "requests.jsonl", folder / "manifest.json"
    if meta_file.exists():
        meta = json.loads(meta_file.read_text())
        raw = data_file.read_bytes()
        if meta["day"] != day or meta["plan_id"] != plan_id:
            raise ValueError(f"{day}: wrong saved manifest")
        if hashlib.sha256(raw).hexdigest() != meta["sha256"] or len(raw.splitlines()) != meta["rows"]:
            raise ValueError(f"{day}: saved data changed")
        return meta, "verified"

    rows, where, expected = fetch_day(day)
    content = "".join(json.dumps(row, sort_keys=True) + "\n" for row in rows).encode("utf-8")
    meta = {"plan_id": plan_id, "day": day, "source": plan["source_url"],
            "where": where, "fields": plan["fields"], "page_size": page_size,
            "order": "created_date ASC, unique_key ASC", "rows": expected,
            "sha256": hashlib.sha256(content).hexdigest(),
            "fetched_at_utc": datetime.now(timezone.utc).isoformat()}
    folder.mkdir(parents=True, exist_ok=True)
    pending = folder / "requests.jsonl.pending"
    pending.write_bytes(content)
    if hashlib.sha256(pending.read_bytes()).hexdigest() != meta["sha256"]:
        raise ValueError(f"{day}: write verification failed")
    pending.replace(data_file)
    meta_pending = folder / "manifest.json.pending"
    meta_pending.write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")
    meta_pending.replace(meta_file)
    return meta, "saved"

print("Resumable day saver ready")

In [ ]:
# Cell 6 — Why: derive every date from the saved plan, not memory.
start = date.fromisoformat(plan["windows_start_inclusive_end_exclusive"]["train"][0])
end = date.fromisoformat(plan["windows_start_inclusive_end_exclusive"]["test"][1])
days = [(start + timedelta(days=i)).isoformat() for i in range((end - start).days)]
if len(days) != plan["daily_partitions_planned"] or len(days) != len(set(days)):
    raise ValueError("Plan dates are incomplete or duplicated")
print("First day:", days[0], "Last day:", days[-1], "Total:", len(days))

In [ ]:
# Cell 7 — Why: make bounded progress; reruns skip saved days.
deadline = time.monotonic() + 10 * 60
processed = 0
skipped = 0
for day in days:
    if time.monotonic() >= deadline:
        print("Time budget reached. Rerun this cell to continue.")
        break
    if (daily_root / day / "manifest.json").exists():
        skipped += 1
        continue
    meta, action = save_day(day)
    processed += 1
    print(day, action, meta["rows"], "rows", flush=True)
print("New days this run:", processed)
print("Previously saved days skipped:", skipped)

In [ ]:
# Cell 8 — Why: show coverage without opening test outcomes.
complete = [day for day in days if (daily_root / day / "manifest.json").exists()]
missing = [day for day in days if day not in set(complete)]
print("Completed days:", len(complete), "of", len(days))
print("Remaining days:", len(missing))
print("Next missing day:", missing[0] if missing else "none")
print("Test outcomes remain sealed for model selection")

In [ ]:
# Cell 9 — Why: create the full dataset ID only when all days exist.
if missing:
    print("Dataset not finalized yet; rerun Cell 7 later.")
else:
    parts = []
    for number, day in enumerate(days, 1):
        folder = daily_root / day
        meta = json.loads((folder / "manifest.json").read_text())
        raw_file = folder / "requests.jsonl"
        if hashlib.sha256(raw_file.read_bytes()).hexdigest() != meta["sha256"]:
            raise ValueError(f"{day}: final verification failed")
        parts.append({"day": day, "file": str(raw_file.relative_to(root)),
                      "sha256": meta["sha256"], "rows": meta["rows"]})
        if number % 25 == 0 or number == len(days):
            print("Verified files:", number, "of", len(days), flush=True)
    dataset_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    dataset_dir = root / "datasets" / dataset_id
    dataset_dir.mkdir(parents=True, exist_ok=False)
    dataset = {"dataset_id": dataset_id, "created_at_utc": datetime.now(timezone.utc).isoformat(),
               "source_plan_id": plan_id, "partitions": parts,
               "total_rows": sum(part["rows"] for part in parts),
               "sampling_rule": "all requests from every day in the saved 485-day plan",
               "lesson_version": "stage9b-v1"}
    (dataset_dir / "manifest.json").write_text(json.dumps(dataset, indent=2) + "\n", encoding="utf-8")
    print("Complete dataset ID:", dataset_id, "Rows:", dataset["total_rows"])

# Stage 9B takeaway and quiz

**What you learned:** The collector follows a saved plan, counts and pages through each day, checks IDs and dates, records query details and a checksum, and safely resumes after interruptions. The final dataset ID appears only after all 485 daily files pass verification.

**Easy quiz:**

1. Why do we count a day both before and after paging?
2. Why do we order pages by creation time and request ID?
3. What happens on a rerun when a day's manifest already exists, and when is its checksum checked again?
4. Why is the full dataset ID created only after all 485 days are complete?

Send me **New days this run**, **Completed days**, **Remaining days**, **Next missing day**, and any first error. Cell 7 may take up to about 10 minutes by design; progress prints after each completed day.